# Notebook 2 — Drawdown-Penalised PPO on DOGEUSDT Market Replay

Implements Falces Marin's §7 suggestion: penalise drawdowns in the reward.

    r_t = ΔPnL − φq²Δt − α · drawdown_t

Compares A-S baseline vs drawdown-penalised PPO.
This serves as the ablation baseline for Notebook 3 (CVaR constraint).

In [1]:
import sys, pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize

from procs.stochastic_processes import (
    MarketReplayMidpriceModel, PoissonArrivalModel, ExponentialFillFunction,
)
from procs.gym.model_dynamics import LimitOrderModelDynamics
from procs.gym.trading_environment import TradingEnvironment
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.rewards import PnLReward, CjMmDrawdownPenalty
from procs.agents import AvellanedaStoikovAgent, Sb3Agent
from procs.gym.helpers.plotting import plot_trajectory, plot_learned_policy
from procs.gym.helpers.generate_trajectory_stats import generate_trajectory_stats
from procs.gym.helpers.fast_rollout import fast_simulate
from procs.gym.data_loader import load_single_day
from procs.gym.calibration import tune_gamma
from procs.gym.features import FeatureComputer, RollingVolatility
from procs.gym.reward_scale import estimate_reward_scale

%matplotlib inline


## 1. Load Data

In [2]:
DATA_PATH = r"C:\Users\john-\Documents\Thesis_AI4T\datasets\binance_book_snapshot_25_2025-01-01_DOGEUSDT.csv"
S, dt_sec, dt_index = load_single_day(DATA_PATH)
T_sec = float(dt_sec.sum())
sigma = MarketReplayMidpriceModel(S, dt_sec).volatility
print(f"Loaded {len(S):,} snapshots, σ={sigma:.6f}, T={T_sec:.0f}s")

Loaded 713,815 snapshots, σ=0.000021, T=86398s


## 2. Parameters & Calibrate γ

In [ ]:
kappa, A, tick, Q_MAX, phi = 35_000, 0.8, 0.00001, 10, 0.01
alpha_dd = 1.0   # drawdown penalty weight (sweep later)

# Calibrate γ
best_gamma, study = tune_gamma(
    midprices=S, dt_array=dt_sec,
    sigma=sigma, kappa=kappa, A=A,
    tick_size=tick, Q_MAX=Q_MAX,
    gamma_range=(0.001, 1.0),
    n_trials=20, num_trajectories=50,
)
print(f"Using γ = {best_gamma:.6f}")

reward_scale = estimate_reward_scale(
    midprices=S, dt_array=dt_sec, sigma=sigma,
    kappa=kappa, A=A, terminal_time=T_sec,
    tick_size=tick, Q_MAX=Q_MAX,
    num_trajectories=50, use_bm=False,
)
print(f"Reward scale: {reward_scale:.4f}")

## 3. A-S Baseline

In [ ]:
as_agent = AvellanedaStoikovAgent(best_gamma, sigma, kappa, T_sec, tick)

# Single trajectory with timestamps
env_as = TradingEnvironment(
    model_dynamics=LimitOrderModelDynamics(
        midprice_model=MarketReplayMidpriceModel(S, dt_sec, 1),
        arrival_model=PoissonArrivalModel(np.array([A, A]), 1, use_linear_approximation=False),
        fill_probability_model=ExponentialFillFunction(kappa, 1),
    ),
    reward_function=PnLReward(), max_inventory=Q_MAX,
)
plot_trajectory(env_as, as_agent, seed=42, datetime_index=dt_index)

# Metrics
as_stats = fast_simulate(
    midprices=S, dt_array=dt_sec, gamma=best_gamma, sigma=sigma,
    kappa=kappa, A=A, terminal_time=T_sec, tick_size=tick, Q_MAX=Q_MAX,
    num_trajectories=50, seed=42, use_linear_approximation=False,
)
print(f"A-S Sharpe: {as_stats['sharpe'].mean():.4f}")
print(f"A-S Max DD: {as_stats['max_drawdown'].mean():.4f}")
print(f"A-S PnL:    {as_stats['total_pnl'].mean():.4f}")

## 4. Train Drawdown-Penalised PPO

Using ``CjMmDrawdownPenalty(φ, α)``: r_t = ΔPnL − φq²Δt − α·drawdown_t

In [5]:
# get_doge_env always returns a raw TradingEnvironment (no manual normalisation).
# VecNormalize wraps the SB3 env at training time and is saved/loaded for eval.
# Features are always included so train and eval obs dims always match.

def get_doge_env(reward_fn=None):
    """Raw TradingEnvironment with RollingVolatility feature (obs dim=5)."""
    if reward_fn is None:
        reward_fn = PnLReward()
    return TradingEnvironment(
        model_dynamics=LimitOrderModelDynamics(
            midprice_model=MarketReplayMidpriceModel(S, dt_sec, 1),
            arrival_model=PoissonArrivalModel(
                np.array([A, A]), 1, use_linear_approximation=False,
            ),
            fill_probability_model=ExponentialFillFunction(kappa, 1),
        ),
        reward_function=reward_fn,
        max_inventory=Q_MAX,
        feature_computer=FeatureComputer([RollingVolatility(100)]),
        # No manual normalisation — VecNormalize handles scaling adaptively
    )


def make_vecnorm(sb3_env, training=True):
    """Wrap with VecNormalize. clip_obs=10 prevents outlier snapshots
    from distorting running statistics. norm_reward=True stabilises
    value function training (same benefit as reward_scale, but adaptive)."""
    return VecNormalize(
        sb3_env,
        norm_obs=True,
        norm_reward=True,
        clip_obs=10.0,
        training=training,   # False at eval time: stats frozen
    )


In [ ]:
%%time
train_env = get_doge_env(
    reward_fn=CjMmDrawdownPenalty(per_step_inventory_aversion=phi, drawdown_penalty=alpha_dd),
)
sb3_raw = StableBaselinesTradingEnvironment(train_env)
sb3_env = make_vecnorm(sb3_raw, training=True)   # adaptive obs + reward scaling

model_dd = PPO(
    "MlpPolicy", sb3_env, verbose=1, device="cpu",
    n_steps=2048, batch_size=512, n_epochs=10,
    learning_rate=3e-4, gamma=0.999, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.01,
    tensorboard_log="./tb_logs_dd",
)
model_dd.learn(total_timesteps=2048 * 50)
print("Drawdown-penalised PPO trained.")

# Save VecNormalize stats alongside the model — needed for correct evaluation
model_dd.save("../models/ppo_dd_penalised_doge")
sb3_env.save("../models/vecnorm_dd.pkl")
print("Model + VecNorm stats saved.")

## 5. Compare A-S vs DD-Penalised PPO

In [ ]:
# Load VecNorm stats and freeze (training=False) so running stats don't shift
eval_raw = StableBaselinesTradingEnvironment(get_doge_env(reward_fn=PnLReward()))
eval_env_vn = VecNormalize.load("../models/vecnorm_dd.pkl", eval_raw)
eval_env_vn.training = False       # freeze stats
eval_env_vn.norm_reward = False    # evaluate on raw PnL, not scaled reward

# Sb3Agent with VecNormalize: pass vecnorm_env so agent normalises obs correctly
ppo_dd_agent = Sb3Agent(model_dd, vecnorm_env=eval_env_vn)

# Evaluate on raw env — Sb3Agent applies VecNorm obs scaling internally
eval_env = get_doge_env(reward_fn=PnLReward())
ppo_stats = generate_trajectory_stats(eval_env, ppo_dd_agent, seed=42)

comparison = pd.DataFrame({
    "A-S Baseline": {
        "Sharpe":     as_stats["sharpe"].mean(),
        "Sortino":    as_stats["sortino"].mean(),
        "Max DD":     as_stats["max_drawdown"].mean(),
        "P&L-to-MAP": as_stats["pnl_to_map"].mean(),
        "Final PnL":  as_stats["total_pnl"].mean(),
    },
    "DD-Penalised PPO": {
        "Sharpe":     ppo_stats["sharpe"][0],
        "Sortino":    ppo_stats["sortino"][0],
        "Max DD":     ppo_stats["max_drawdown"][0],
        "P&L-to-MAP": ppo_stats["pnl_to_map"][0],
        "Final PnL":  ppo_stats["total_pnl"][0],
    },
}).T
print("DOGEUSDT Comparison:")
print(comparison.to_string(float_format="%.6f"))

In [ ]:
# α sweep: shows the penalty-vs-PnL tradeoff that the CMDP approach avoids.
# This is the key thesis comparison: no single α matches the CVaR agent's tradeoff.
print(f"{'α':>6}  {'Sharpe':>8}  {'MaxDD':>8}  {'PnL':>8}")
print("-" * 38)

for alpha in [0.1, 1.0, 10.0]:
    env_a = get_doge_env(reward_fn=CjMmDrawdownPenalty(phi, drawdown_penalty=alpha))
    sb3_a  = make_vecnorm(StableBaselinesTradingEnvironment(env_a), training=True)

    m = PPO("MlpPolicy", sb3_a, verbose=0, device="cpu",
            n_steps=2048, batch_size=512, n_epochs=10,
            learning_rate=3e-4, gamma=0.999, gae_lambda=0.95,
            clip_range=0.2, ent_coef=0.01)
    m.learn(total_timesteps=2048 * 30)

    # Freeze VecNorm for evaluation
    ev_raw = StableBaselinesTradingEnvironment(get_doge_env())
    ev_vn  = VecNormalize(ev_raw, norm_obs=True, norm_reward=False,
                          clip_obs=10.0, training=False)
    ev_vn.obs_rms = sb3_a.obs_rms   # copy running stats from training
    ev_vn.ret_rms = sb3_a.ret_rms

    ag = Sb3Agent(m, vecnorm_env=ev_vn)
    ev = get_doge_env()
    st = generate_trajectory_stats(ev, ag, seed=42)
    print(f"  {alpha:>4.1f}  {st['sharpe'][0]:>8.4f}  {st['max_drawdown'][0]:>8.4f}  {st['total_pnl'][0]:>8.4f}")


In [ ]:
print("Model and VecNorm stats already saved during training:")
print("  ../models/ppo_dd_penalised_doge.zip")
print("  ../models/vecnorm_dd.pkl")